In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import ast

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder


from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import train_test_split
import shap

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import umap
from sklearn.manifold import TSNE


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF


In [ ]:
def add_onehot_encoding(df, column, drop_original=False, drop_first=False, prefix=None):
    """
    Adds one-hot encoded columns for the specified categorical column.
    
    Args:
        df: pandas DataFrame
        column: str, the column name to one-hot encode
        drop_original: bool, if True, drop the original column after encoding
        drop_first: bool, if True, drop the first level (to avoid collinearity)
        prefix: str or None, prefix for new columns; defaults to the column name

    Returns:
        df with one-hot encoded columns added
    """
    dummies = pd.get_dummies(df[column], prefix=prefix or column, drop_first=drop_first)
    df = pd.concat([df, dummies], axis=1)
    if drop_original:
        df = df.drop(columns=[column])
    return df


def add_label_encoding(df, column, new_col=None, inplace=False):
    """
    Adds a label-encoded version of a categorical column.
    
    Args:
        df: pandas DataFrame
        column: str, the column to encode
        new_col: str or None, the name for the new encoded column (defaults to column + '_label')
        inplace: bool, if True, modifies df in-place

    Returns:
        df with the label-encoded column added
    """
    le = LabelEncoder()
    encoded = le.fit_transform(df[column].astype(str))
    colname = new_col or f"{column}_label"
    if inplace:
        df[colname] = encoded
        return df
    else:
        df_out = df.copy()
        df_out[colname] = encoded
        return df_out
    

def cluster_entities(df_entity, name_col, feature_cols, max_clusters=10, cluster_size_target=10, min_entities_for_clustering=3):
    n_entities = len(df_entity)
    if feature_cols is None or n_entities < 2:
        df_entity[f'{name_col}_cluster'] = 0
        return df_entity
    can_cluster = n_entities >= min_entities_for_clustering
    if not can_cluster:
        df_entity[f'{name_col}_cluster'] = 0
        return df_entity
    n_clusters = min(max_clusters, max(2, n_entities // cluster_size_target))
    X = df_entity[feature_cols].fillna(0).values
    if n_clusters < 2:
        df_entity[f'{name_col}_cluster'] = 0
    else:
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        df_entity[f'{name_col}_cluster'] = kmeans.fit_predict(X)
    return df_entity


def get_main_genre(x):
    mode = x.mode()
    return mode.iloc[0] if not mode.empty else None

def get_sub_genre(x):
    mode = x.mode()
    return mode.iloc[1] if len(mode) > 1 else None

def get_df_entity(
    df,
    entity_col,
    name_col,
    popularity_col,
    avg_rating_col,
    main_genre_col='main_genre_id',
    sub_genre_col='main_genre_id',
    count_col='movieId',
    additional_aggs=None,
    cluster_features=None,
    max_clusters=15,
    cluster_size_target=10,
    min_entities_for_clustering=3
):
    aggs = {
        entity_col: (entity_col, 'first'),
        name_col: (name_col, 'first'),
        "popularity": (popularity_col, 'first'),
        'num_movies': (count_col, 'count'),
        "main_genre_id": (main_genre_col, get_main_genre),
        "sub_genre_id": (sub_genre_col, get_sub_genre),
        "avg_rating": (avg_rating_col, 'mean'),
        'total_vote_count': ('vote_count', 'sum'),
        'avg_popularity_score': ('popularity_score', 'mean'),
        'avg_critical_success': ('critical_success', 'mean'),
    }
    if additional_aggs:
        aggs.update(additional_aggs)
    df_entity = df.groupby(entity_col).agg(**aggs).reset_index(drop=True)
    cols = df_entity.columns.tolist()
    cols.insert(0, cols.pop(cols.index(name_col)))
    cols.insert(0, cols.pop(cols.index(entity_col)))
    df_entity = df_entity[cols].sort_values(
        ["popularity", "num_movies", "avg_popularity_score"], ascending=False
    )
    if cluster_features is not None:
        df_entity = cluster_entities(
            df_entity,
            name_col=name_col,
            feature_cols=cluster_features,
            max_clusters=max_clusters,
            cluster_size_target=cluster_size_target,
            min_entities_for_clustering=min_entities_for_clustering
        )
    else:
        df_entity[f'{name_col}_cluster'] = 0
    return df_entity


def get_df_genres(df, id2genre):
    df_exploded = df.explode('genre_id_list').rename(columns={'genre_id_list': 'genre_id'})
    director_counts = df_exploded.groupby(['genre_id', 'director_id']).size().reset_index(name='num_movies')
    top2_directors = director_counts.groupby('genre_id').apply(
        lambda group: pd.Series({
            'top_director_id': group.sort_values('num_movies', ascending=False)['director_id'].iloc[0] if len(group) > 0 else None,
            'sub_director_id': group.sort_values('num_movies', ascending=False)['director_id'].iloc[1] if len(group) > 1 else None
        })
    ).reset_index()

    actor_counts = df_exploded.groupby(['genre_id', 'actor_id']).size().reset_index(name='num_movies')
    top2_actors = actor_counts.groupby('genre_id').apply(
        lambda group: pd.Series({
            'top_lead_actor_id': group.sort_values('num_movies', ascending=False)['actor_id'].iloc[0] if len(group) > 0 else None,
            'sub_lead_actor_id': group.sort_values('num_movies', ascending=False)['actor_id'].iloc[1] if len(group) > 1 else None
        })
    ).reset_index()

    idx = df_exploded.groupby('genre_id')['vote_average'].idxmax()
    top_movie_ids = df_exploded.loc[idx, ['genre_id', 'movieId']].drop_duplicates('genre_id').set_index('genre_id')

    df_genres = (
        df_exploded.groupby('genre_id').agg(
            num_movies=('movieId', 'count'),
            avg_rating=('vote_average', 'mean'),
            total_vote_count=('vote_count', 'sum'),
            avg_popularity_score=('popularity_score', 'mean'),
            avg_critical_success=('critical_success', 'mean')
        ).reset_index()
        .merge(top2_directors, on='genre_id', how='left')
        .merge(top2_actors, on='genre_id', how='left')
    )
    df_genres = df_genres.merge(
        top_movie_ids[['movieId']], left_on='genre_id', right_index=True, how='left'
    ).rename(columns={'movieId': 'top_movie_id'})
    df_genres['genre'] = df_genres['genre_id'].map(id2genre)
    first_cols = ['genre_id', 'genre']
    df_genres = df_genres[first_cols + [c for c in df_genres.columns if c not in first_cols]]
    return df_genres


def get_genre_id_list(row, genre2id):
    genre_ids = [genre2id[g] for g in row['genre_list'] if g in genre2id]
    if genre_ids:
        return genre_ids
    elif pd.notnull(row['main_genre']):
        return [genre2id[row['main_genre']]]
    else:
        return []


def runtime_bin(runtime):
    if pd.isnull(runtime):
        return 'unknown'
    elif runtime <= 60:
        return 'very_short'
    elif runtime <= 85:
        return 'short'
    elif runtime <= 110:
        return 'medium'
    elif runtime <= 130:
        return 'standard'
    elif runtime <= 160:
        return 'long'
    else:
        return 'epic'



def enrich_movies_with_entities(
    df_movies, 
    df_directors, 
    df_lead_actors, 
    df_genres
):
    directors_merge = df_directors.add_prefix('director_')
    directors_merge = directors_merge.rename(columns={'director_director_id': 'director_id'})
    
    actors_merge = df_lead_actors.add_prefix('actor_')
    actors_merge = actors_merge.rename(columns={'actor_actor_id': 'actor_id'})
    
    genres_merge = df_genres.add_prefix('genre_')
    genres_merge = genres_merge.rename(columns={'genre_genre_id': 'main_genre_id'})
    
    df_enriched = df_movies.merge(directors_merge, on='director_id', how='left', suffixes=('', '_director'))
    df_enriched = df_enriched.merge(actors_merge, on='actor_id', how='left', suffixes=('', '_actor'))
    df_enriched = df_enriched.merge(genres_merge, on='main_genre_id', how='left', suffixes=('', '_genre'))
    
    return df_enriched


def add_clusters(
    df, 
    feature_cols,
    cluster_col_prefix,
    n_clusters=10,
    min_cluster_size=5,
    pca_components=None,
    random_state=42
):
    df = df.copy()
    X = df[feature_cols].fillna(0).values

    if X.shape[1] > 5:
        pca_n = min(X.shape[0], X.shape[1], pca_components or 10)
        pca = PCA(n_components=pca_n, random_state=random_state)
        X = pca.fit_transform(X)
    else:
        pca = None

    n_entities = X.shape[0]
    if n_entities < min_cluster_size:
        cluster_labels = np.zeros(n_entities, dtype=int)
    else:
        n_clust = min(n_clusters, n_entities) if n_entities >= n_clusters else max(2, n_entities // 2)
        kmeans = KMeans(n_clusters=n_clust, random_state=random_state, n_init=10)
        cluster_labels = kmeans.fit_predict(X)

    df[f"{cluster_col_prefix}_cluster"] = cluster_labels
    return df


def extract_topics_from_text(df, col, prefix, max_features=20, n_topics=20, random_state=42):
    tfidf = TfidfVectorizer(max_features=max_features, stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df[col].fillna(''))
    nmf = NMF(n_components=n_topics, random_state=random_state)
    topics = nmf.fit_transform(tfidf_matrix)
    topic_cols = [f"{prefix}_topic_{i}" for i in range(n_topics)]
    topics_df = pd.DataFrame(topics, columns=topic_cols, index=df.index)
    feature_names = tfidf.get_feature_names_out()
    topic_words = {idx: [feature_names[i] for i in topic.argsort()[-8:][::-1]]
                   for idx, topic in enumerate(nmf.components_)}
    return topics_df, topic_words


def extract_secondary_genres(genre_list):
    if isinstance(genre_list, (list, np.ndarray)):
        genre_list = list(genre_list)
        genres = genre_list[:2] + [None] * (2 - len(genre_list))
    elif genre_list is None:
        genres = [None, None]
    else:
        try:
            genres_eval = eval(genre_list) if isinstance(genre_list, str) and genre_list.startswith('[') else str(genre_list).split(',')
            genres_eval = [g.strip() for g in genres_eval if g.strip()]
            genres = genres_eval[:2] + [None] * (2 - len(genres_eval))
        except Exception:
            genres = [None, None]
    return pd.Series(genres, index=['secondary_genre_1', 'secondary_genre_2'])

def create_genre_str(df):
    df[['secondary_genre_1', 'secondary_genre_2']] = df['genre_list'].apply(extract_secondary_genres)
    df['genre_str'] = df.apply(
        lambda row: ' '.join([
            str(g) for g in [row['secondary_genre_1'], row['secondary_genre_2']]
            if g is not None and str(g).strip().lower() != 'none'
        ]),
        axis=1
    )
    return df

def robust_tags(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    elif pd.isna(x):
        return []
    else:
        return [str(x)]
    

def join_list_for_tfidf(tags):
    """Join array or list of tags into a single string for TF-IDF."""
    if isinstance(tags, (list, np.ndarray)):
        return ' '.join([str(tag) for tag in tags if tag is not None])
    elif pd.isnull(tags):
        return ''
    else:
        return str(tags)


def cluster_movies(df, features, n_clusters=20, pca_n=15, cluster_col='movie_cluster', random_state=42):
    X = df[features].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca = PCA(n_components=min(pca_n, X_scaled.shape[1]), random_state=random_state)
    X_pca = pca.fit_transform(X_scaled)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    df[cluster_col] = kmeans.fit_predict(X_pca)
    return df, kmeans, pca, scaler



# File Read


In [ ]:
df_movies = pd.read_parquet("../data/movies_filtered_cleaned.parquet")
df_ratings = pd.read_parquet("../data/processed_df_ratings_filtered.parquet")
df_ratings = df_ratings[['userId', 'movieId', 'rating']]

df_users = pd.read_parquet("../data/all_users_stats_post_movies_filter.parquet")

# Dict and info setup


In [ ]:
all_genres_from_list = set(g for genres in df_movies['genre_list'] for g in genres)
all_genres_from_main = set(df_movies['main_genre'].dropna().unique())
all_genres = all_genres_from_list | all_genres_from_main
genre2id = {genre: idx for idx, genre in enumerate(sorted(all_genres))}
id2genre = {idx: genre for genre, idx in genre2id.items()}

all_directors = df_movies['director'].dropna().unique()
director2id = {director: idx for idx, director in enumerate(sorted(all_directors))}
id2director = {idx: director for director, idx in director2id.items()}

all_actors = df_movies['lead_actor'].dropna().unique()
actor2id = {actor: idx for idx, actor in enumerate(sorted(all_actors))}
id2actor = {idx: actor for actor, idx in actor2id.items()}

df_movies['main_genre_id'] = df_movies['main_genre'].map(genre2id)

df_movies['genre_id_list'] = df_movies.apply(get_genre_id_list, axis=1, args=(genre2id,))

df_movies['director_id'] = df_movies['director'].map(director2id)
df_movies['actor_id'] = df_movies['lead_actor'].map(actor2id)

# Filtered DF setup


In [ ]:
filtered_df = df_movies.copy()
filtered_df.drop(columns="crowd_approval", inplace = True)

filtered_df['runtime_bin'] = filtered_df['runtime'].apply(runtime_bin)

runtime_bin2id = {
    'very_short': 0,
    'short': 1,
    'medium': 2,
    'standard': 3,
    'long': 4,
    'epic': 5,
    'unknown': -1
}
filtered_df['runtime_bin_id'] = filtered_df['runtime_bin'].map(runtime_bin2id)

id_cols = ['director_id', 'actor_id', 'main_genre_id', 'genre_id_list', 'runtime_bin_id']

id_cols = [c for c in id_cols if c in filtered_df.columns]
other_cols = [c for c in filtered_df.columns if c not in id_cols]

filtered_df = filtered_df[id_cols + other_cols]
filtered_df = add_onehot_encoding(filtered_df, 'runtime_bin')
filtered_df = add_onehot_encoding(filtered_df, 'main_genre')

filtered_df[['secondary_genre_1', 'secondary_genre_2']] = filtered_df['genre_list'].apply(extract_secondary_genres)

In [ ]:
filtered_df['overview'] = filtered_df['overview'].fillna('')
topic_overview_df, topic_words = extract_topics_from_text(filtered_df, col='overview', prefix='overview',max_features=30, n_topics=20)
filtered_df = pd.concat([filtered_df, topic_overview_df], axis=1)


filtered_df['tag_str'] = filtered_df['tag'].apply(join_list_for_tfidf)

tag_topic_df, tag_topic_words = extract_topics_from_text(
    filtered_df, 
    col='tag_str', 
    prefix='tag', 
    max_features=20,
    n_topics=10
)
filtered_df = pd.concat([filtered_df, tag_topic_df], axis=1)


filtered_df['keywords_str'] = filtered_df['keywords'].apply(join_list_for_tfidf)

keywords_topic_df, keywords_topic_words = extract_topics_from_text(
    filtered_df, 
    col='keywords_str', 
    prefix='keywords', 
    max_features=30,
    n_topics=15
)
filtered_df = pd.concat([filtered_df, keywords_topic_df], axis=1)


topic_prefixes = ["overview_topic_", "tag_topic_", "keywords_topic_"]

all_topic_cols = [
    col for col in filtered_df.columns
    if any(col.startswith(prefix) for prefix in topic_prefixes)
]

In [ ]:
director_topic_means = filtered_df.groupby('director_id')[all_topic_cols].mean().reset_index()
actor_topic_means = filtered_df.groupby('actor_id')[all_topic_cols].mean().reset_index()

In [ ]:
df_directors = get_df_entity(
    df=filtered_df,
    entity_col='director_id',
    name_col='director',
    popularity_col='director_popularity',
    avg_rating_col='vote_average',
    cluster_features=None
)

df_directors = df_directors.merge(director_topic_means, on='director_id', how='left')
cluster_features = (
    ['popularity', 'num_movies', 'avg_popularity_score', 'avg_critical_success']
    + all_topic_cols
)

df_directors = cluster_entities(
    df_entity=df_directors,
    name_col='director',
    feature_cols=cluster_features,
    max_clusters=8,
    cluster_size_target=10,
    min_entities_for_clustering=5
)

In [ ]:
# df_directors["director_cluster"].value_counts().sum()

In [ ]:
actor_topic_means = filtered_df.groupby('actor_id')[all_topic_cols].mean().reset_index()

df_lead_actors = get_df_entity(
    df=filtered_df,
    entity_col='actor_id',
    name_col='lead_actor',
    popularity_col='lead_actor_popularity',
    avg_rating_col='vote_average',
    cluster_features=None
)
df_lead_actors = df_lead_actors.merge(actor_topic_means, on='actor_id', how='left')
cluster_features_actors = (
    ['popularity', 'num_movies', 'avg_popularity_score', 'avg_critical_success']
    + all_topic_cols
)
df_lead_actors = cluster_entities(
    df_entity=df_lead_actors,
    name_col='lead_actor',
    feature_cols=cluster_features_actors,
    max_clusters=8,
    cluster_size_target=10,
    min_entities_for_clustering=5
)

# Enrich df movies with all info about directors, actors and genres.


In [ ]:
df_genres = get_df_genres(filtered_df, id2genre)

In [ ]:
filtered_df = enrich_movies_with_entities(
    filtered_df, df_directors, df_lead_actors, df_genres
)

filtered_df.rename(columns = {"director_director_cluster": "director_cluster", "actor_lead_actor_cluster": "actor_cluster"}, inplace = True)
filtered_df.drop(columns= ["actor_lead_actor", "director_director", "genre_genre", "director_popularity_director", "actor_popularity"], inplace = True)

In [ ]:
filtered_df.info(memory_usage="deep")

# Movie clustering - complex


In [ ]:
core_num_columns = [
    'popularity',                # TMDB/aggregated popularity score
    'vote_average',              # Average user rating
    'vote_count',                # How many votes (indicator of reach)
    'critical_success',
    'runtime_bin_id',
    'release_decade',
    # 'release_year',
    "main_genre_id"

] # + [col for col in filtered_df.columns if col.startswith('main_genre_')]
overview_topic_cols = [col for col in filtered_df.columns if col.startswith("overview_topic_")]
tag_topic_cols = [col for col in filtered_df.columns if col.startswith("tag_topic_")]
keywords_topic_cols = [col for col in filtered_df.columns if col.startswith("keyword_topic_")]
core_num_columns += [col for col in filtered_df.columns if col.startswith('runtime_bin_')]

core_num_columns += overview_topic_cols
# core_num_columns += tag_topic_cols
core_num_columns += keywords_topic_cols

core_num_columns += [
    'lead_actor_popularity',      # The "star" power of the lead actor
    'director_popularity',        # The "star" power of the director
    # 'director_avg_rating',        # How well their movies do on average
    # 'actor_avg_rating',
    # 'director_avg_popularity_score',
    # 'actor_avg_popularity_score',
    # 'director_num_movies',        # Experience/size of body of work
    # 'actor_num_movies',
    # 'director_total_vote_count',  # Reach of director's movies
    # 'actor_total_vote_count',
    # 'director_avg_critical_success',
    # 'actor_avg_critical_success',
]

core_num_columns += [
    'director_cluster', 
    'actor_cluster',
]

In [ ]:
X = filtered_df[core_num_columns].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_components = min(25, X_scaled.shape[1], X_scaled.shape[0]-1)  # or 'mle' if many rows
pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X_scaled)
filtered_df['pca_1'] = X_pca[:, 0]
filtered_df['pca_2'] = X_pca[:, 1]

In [ ]:
min_clusters = 2
max_clusters = 8
n_rows = X_pca.shape[0]
n_clusters = min(max_clusters, max(min_clusters, n_rows // 15))

In [ ]:
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
filtered_df['movie_cluster'] = kmeans.fit_predict(X_pca)

In [ ]:
filtered_df

In [ ]:
# filtered_df.to_parquet("1_movies_data_for_app.parquet", index = False)

In [ ]:
# STOP CODE

# Graphs


In [ ]:
scatter_df = filtered_df.copy()
scatter_df['pca_1'] = X_pca[:, 0]
scatter_df['pca_2'] = X_pca[:, 1]

fig = px.scatter(
    scatter_df, x='pca_1', y='pca_2',
    color='movie_cluster',
    hover_data=['title', 'main_genre', 'director', 'lead_actor', 'vote_average', 'popularity_score'],
    title="Movie Clusters in PCA Space"
)
fig.show()

# Recommendation system


In [ ]:
df_users['top_genre_1_id'] = df_users['top_genre_1'].map(genre2id)
df_users['top_genre_2_id'] = df_users['top_genre_2'].map(genre2id)
df_users['top_genre_3_id'] = df_users['top_genre_3'].map(genre2id)


user_cluster_features = [
    'num_ratings', 'mean_rating', 'std_rating', 'genre_diversity',
    'top_genre_1_id', 'top_genre_2_id', 'top_genre_3_id'
]

X = df_users[user_cluster_features].fillna(0).values

n_users = len(df_users)
n_clusters = min(25, max(2, n_users // 100))

if n_users >= 3:
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    df_users['user_cluster'] = kmeans.fit_predict(X)
else:
    df_users['user_cluster'] = 0
df_users.to_parquet("../data/all_users_stats_with_clusters.parquet")

In [ ]:
def get_movie_pool(filtered_df, genre_ids=None, year_range=None, director_ids=None):
    df = filtered_df
    if genre_ids is not None:
        df = df[df['main_genre_id'].isin(genre_ids)]
    if year_range is not None:
        df = df[(df['release_year'] >= year_range[0]) & (df['release_year'] <= year_range[1])]
    if director_ids is not None:
        df = df[df['director_id'].isin(director_ids)]
    return df


In [ ]:
def recommend_movies_for_user(
    user_id,
    filtered_df,
    df_ratings,
    df_users,
    n_recs=5,
    explain=True,
    overview_topic_cols=None,
    topic_words=None,
    tag_topic_cols=None,
    tag_topic_words=None,
):
    seen = df_ratings[df_ratings['userId'] == user_id]['movieId'].tolist()
    pool = filtered_df[~filtered_df['movieId'].isin(seen)].copy()
    if pool.empty:
        return []

    user_row = df_users[df_users['userId'] == user_id]
    user_top_genres = [user_row['top_genre_1_id'].values[0], user_row['top_genre_2_id'].values[0]]

    pool['score'] = 0
    pool.loc[pool['main_genre_id'].isin(user_top_genres), 'score'] += 1
    if 'movie_cluster' in pool.columns and 'user_cluster' in user_row.columns:
        pool.loc[pool['movie_cluster'] == user_row['user_cluster'].values[0], 'score'] += 1
    pool['score'] += pool['popularity_score'].rank(pct=True)
    pool['score'] += pool['critical_success'].rank(pct=True)
    pool['score'] += pool['vote_average'].rank(pct=True)

    pool = pool.sort_values('score', ascending=False)
    recs = pool.head(n_recs)

    explanations = []
    for _, row in recs.iterrows():
        why = []
        if row['main_genre_id'] in user_top_genres:
            why.append(f"Matches your favorite genre ({row['main_genre']})")
        if 'movie_cluster' in pool.columns and row['movie_cluster'] == user_row['user_cluster'].values[0]:
            why.append(f"In your preferred cluster (based on similar movies)")
        if row['popularity_score'] > pool['popularity_score'].median():
            why.append("Popular among other users")
        if row['critical_success'] > pool['critical_success'].median():
            why.append("Critically acclaimed")
        # Top topics/keywords/themes
        topic_strs = []
        if overview_topic_cols is not None and topic_words is not None:
            theme_words = get_top_topic_words(row, overview_topic_cols, topic_words, n=2)
            if theme_words:
                topic_strs.append("Overview: " + theme_words)
        if tag_topic_cols is not None and tag_topic_words is not None:
            tag_words = get_top_topic_words(row, tag_topic_cols, tag_topic_words, n=2)
            if tag_words:
                topic_strs.append("Tags: " + tag_words)
        if topic_strs:
            why.append("Notable themes: " + " | ".join(topic_strs))
        explanations.append({
            "movieId": row['movieId'],
            "title": row['title'],
            "explanation": "; ".join(why)
        })

    if explain:
        return explanations
    else:
        return recs[['movieId', 'title']]

def get_top_topic_words(row, topic_cols, topic_words, n=2):
    """Return the most salient topics/words for this row."""
    # Get the topic indices sorted by strength
    topic_strengths = [(i, row[topic_col]) for i, topic_col in enumerate(topic_cols)]
    topic_strengths.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in topic_strengths[:n]]
    words = []
    for idx in top_indices:
        words.extend(topic_words.get(idx, []))
    return ', '.join(words[:8])

In [ ]:
selected_genres = [genre2id['Animation'], genre2id['Comedy']]
filtered_pool = get_movie_pool(filtered_df, genre_ids=selected_genres)

recs = recommend_movies_for_user(
    user_id=4,
    filtered_df=filtered_pool,
    df_ratings=df_ratings,
    df_users=df_users,
    n_recs=5,
    explain=True,
    overview_topic_cols=overview_topic_cols,
    topic_words=topic_words,
    tag_topic_cols=tag_topic_cols,
    tag_topic_words=tag_topic_words
)
for rec in recs:
    print(f"Recommended: {rec['title']}\nWhy: {rec['explanation']}\n")



In [ ]:
tag_topic_words

In [ ]:
df_ratings

In [ ]:
user_id = 4

watched = df_ratings[df_ratings['userId'] == user_id][['movieId', 'rating']]
watched = watched.loc[watched["movieId"].isin(filtered_df["movieId"])]
watched = watched.merge(filtered_pool[['movieId', 'title', 'main_genre', 'pca_1', 'pca_2', 'movie_cluster']], on='movieId', how='inner')
watched['status'] = 'Watched'

recommended_ids = [rec['movieId'] for rec in recs]
recommended = filtered_pool[filtered_pool['movieId'].isin(recommended_ids)].copy()
recommended['status'] = 'Recommended'
recommended['rating'] = recommended["vote_average"].values

plot_df = pd.concat([watched, recommended], ignore_index=True)


plot_df = pd.concat([watched, recommended], ignore_index=True)
# Optionally, add a new column for easier color/symbol mapping
plot_df['status'] = plot_df['status'].astype(str)

PLOT_COLUMNS = [
    'movieId', 'title', 'main_genre', 'rating', 'status', 
    'pca_1', 'pca_2', 'movie_cluster'
]
plot_df = plot_df[PLOT_COLUMNS]

In [ ]:
plot_df.shape

In [ ]:
plot_df_for_vis

In [ ]:
plot_columns = [
    'movieId', 'title', 'main_genre', 'rating', 'status', 
    'pca_1', 'pca_2', 'movie_cluster'
]
plot_df_for_vis = plot_df[plot_columns]

fig = px.scatter(
    plot_df_for_vis,
    x='pca_1', y='pca_2',
    color='status',
    symbol='status',
    size='rating',
    hover_data=['title', 'main_genre', 'rating', 'movie_cluster'],
    # facet_col='status',
    title='Watched & Recommended Movies for User 4 in Feature Space'
)
fig.show()


In [ ]:
bar_df = plot_df_for_vis.copy()
bar_df['User Rated'] = bar_df['status'] == 'Watched'

fig = px.bar(
    bar_df,
    x='main_genre', y='rating', color='status',
    barmode='group',
    title='Watched vs Recommended: Average Rating per Genre'
)
fig.show()

In [ ]:
# filtered_df['pca_1'] = X_pca[:, 0]
# filtered_df['pca_2'] = X_pca[:, 1]
# filtered_df['pca_3'] = X_pca[:, 2]

# fig = px.scatter_3d(
#     filtered_df,
#     x='pca_1',
#     y='pca_2',
#     z='pca_3',
#     color='movie_cluster',
#     opacity=0.8,
#     title='Movie Clusters in 3D PCA Space',
#     hover_data=['title', 'main_genre']
# )
# fig.show()

In [ ]:

# umap_model = umap.UMAP(n_neighbors=25, min_dist=0.2, random_state=42)
# X_umap = umap_model.fit_transform(X_scaled)

# scatter_df['umap_1'] = X_umap[:, 0]
# scatter_df['umap_2'] = X_umap[:, 1]

# fig = px.scatter(
#     scatter_df, x='umap_1', y='umap_2',
#     color='movie_cluster',
#     hover_data=['title', 'main_genre', 'director', 'lead_actor', 'vote_average', 'popularity_score'],
#     title="Movie Clusters in UMAP Space"
# )
# fig.show()


In [ ]:
# tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
# X_tsne = tsne.fit_transform(X_scaled)

# # Prepare DataFrame for plotting
# scatter_df['tsne_1'] = X_tsne[:, 0]
# scatter_df['tsne_2'] = X_tsne[:, 1]

# fig = px.scatter(
#     scatter_df,
#     x='tsne_1', y='tsne_2',
#     color='movie_cluster',
#     title="Movie Clusters in t-SNE Space",
#     opacity=0.8,
#     color_continuous_scale="plasma"
# )
# fig.show()


In [ ]:
STOP CODE